# Layer 2 (FRIENDS), option (a): friendly noise that preserves the shared body

The friendly noise eta is chosen so that the **shared 1024-number embedding** (what every head reads) stays the same with and without eta
(`--friendly_target features`, scale-free MSE). `--friendly_lambda` is the weight of "make eta large": at 1 eta barely leaves its random
start; at 100 it fills its cap.

**Done so far (eps 0.1, seed 0; attack: BWT -9.6, last 57.2):** lambda 1 and 10 at zeta=mu 16/32/64 (BWT -9.9 to -10.2), lambda 100 at 32 (BWT -11.1):
no gain, slightly worse as eta grows. **To run now:** the same at **eps 0.3** (the main table), to report option (a) on the main setting.

## 1. Settings

In [ ]:
EPS = '0.3'                       # '0.3' = the main table (cautious, eps 0.3); '0.1' = the weaker attack
NOISE = {
    '0.3': '/content/drive/MyDrive/brainwash/brainwash-repro/repro/sweep/noise/cifar100_ewc_cautious_eps0.3/noise_ewc_wcur_1.0_cifar100_ewc_cautious_eps0.3__delta_0.3_dataset_split_cifar100_target_task_0_attacked_task_9_noise_optim_lr_0.005__n_iters_1_n_epochs_5000_seed_0_mode_cautious__w_cur_1.0___min_acc_target_29.pkl',
    '0.1': '/content/drive/MyDrive/brainwash/brainwash-repro/repro/sweep/noise/cifar100_ewc_cautious_eps0.1/noise_ewc_wcur_1.0_cifar100_ewc_cautious_eps0.1__delta_0.1_dataset_split_cifar100_target_task_0_attacked_task_9_noise_optim_lr_0.005__n_iters_1_n_epochs_5000_seed_0_mode_cautious__w_cur_1.0___min_acc_target_39.pkl',
}
NOISE_PKL = NOISE[EPS]
# One folder per eps, SHARED by the option a / b / c notebooks: A_clean and B_poison then run only once, and each
# notebook's table shows every arm of that eps. Finished arms (a finished log in this folder) are skipped.
RESULTS_DIR = f'/content/drive/MyDrive/brainwash/layer2_runs/layer2_defense_eps{EPS}'
SEED = 0                            # only seed 0: there are no victim checkpoints for other seeds yet

REPO_URL = 'https://github.com/anushkathagle/brainwash-repro.git'
REPO_BRANCH = 'layer2/Friendly'

## 2. GPU and Google Drive

In [ ]:
import torch, os
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > GPU'
print(torch.__version__, torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount('/content/drive')
assert os.path.isfile(NOISE_PKL), f'not found: {NOISE_PKL}'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('noise pkl:', os.path.basename(NOISE_PKL), f'({os.path.getsize(NOISE_PKL)/1e6:.0f} MB)')
print('results  :', RESULTS_DIR)

## 3. Code: the `layer2/Friendly` branch

The Layer-2 code lives in the repo, in `layer2_defense/` (next to `approaches/` and `repro/`), so nothing is written here.
The first run clones the branch; re-running this cell later **pulls the newest commit** of the branch.
Private repo: add a read-only GitHub token as a Colab secret named `GH_TOKEN` (key icon in the left sidebar).

In [ ]:
import subprocess, sys
REPO_DIR = '/content/brainwash-repro'
url = REPO_URL
try:
    from google.colab import userdata
    tok = userdata.get('GH_TOKEN')
    if tok:
        url = REPO_URL.replace('https://', f'https://{tok}@')
except Exception:
    pass
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '-q', '-b', REPO_BRANCH, url, REPO_DIR], check=True)
else:   # already cloned in this Colab session: move to the newest commit of the branch
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', '-q', url, REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', '-q', '-B', REPO_BRANCH, 'FETCH_HEAD'], check=True)
%cd /content/brainwash-repro
assert os.path.isfile('layer2_defense/friends_layer2.py'), 'layer2_defense/ missing: is REPO_BRANCH right?'
sys.path.insert(0, '/content/brainwash-repro/layer2_defense')
import importlib, colab_helpers as H
importlib.reload(H)
H.record_code_version(RESULTS_DIR)      # which commit produced the runs in this folder -> code_version.txt

## 4. Data and checkpoints (CIFAR-100 and the authors' victim checkpoints, about 200 MB)

In [ ]:
!pip -q install huggingface_hub
!python repro/download_artifacts.py --what cifar100
!python repro/prepare_cifar100.py

## 5. Arms

In [ ]:
def fa(z, w=0, lam=1, comp='both'):   # option (a): eta preserves the shared features; z = zeta = mu (x/255)
    c = '' if comp == 'both' else f' --components {comp}'
    mu = f' --mu_rand {z}' if comp != 'friendly_only' else ''
    return f'--friends --friendly_target features{c} --zeta {z}{mu} --warmup_epochs {w} --friendly_lambda {lam}'

ARMS = [
    # ---- to run now (eps 0.3) ----
    ('A_clean',            ''),                                  # shared with the b / c notebooks (runs once)
    ('B_poison',           '--addnoise'),
    ('a_w0_z32',           '--addnoise ' + fa(32)),              # lambda 1
    ('a_w0_z32_lam10',     '--addnoise ' + fa(32, lam=10)),
    ('a_w0_z64_lam100',    '--addnoise ' + fa(64, lam=100)),     # eta at its cap
    # ---- optional ----
    # ('a_w0_z64_lam100_fr', '--addnoise ' + fa(64, lam=100, comp='friendly_only')),   # eta alone
    # ('aclean_w0_z64_lam100', fa(64, lam=100)),                                         # cost on clean data
    # ---- already done at eps 0.1 (set EPS = '0.1' above to re-run them there) ----
    # ('a_w0_z16' / 'a_w0_z64', ...)  lambda 1:  BWT -10.0 / -9.9,  last 58.6 / 57.9
    # lambda 10, z 16/32/64:          BWT -10.0 / -10.1 / -10.2,  last 58.5 / 58.6 / 58.3
    # lambda 100, z 32:               BWT -11.1,  last 58.7
]
H.run_arms(ARMS, H.base_cmd(NOISE_PKL, RESULTS_DIR, SEED), RESULTS_DIR, SEED)

## Results

In % (as in the BrainWash paper). More negative BWT = more forgetting.
`recovered_%` = how much of the BWT gap between the attack (`ours`) and the clean run an arm closes (100 = as good as clean, negative = worse than no defense).
`last_vs_attack` = last-task accuracy minus the attack's: a negative number is what the recovery costs on the new task.
`old_acc_at_eta` = old-task accuracy when the friendly noise was computed (62.0 = nothing forgotten yet).

In [ ]:
H.results_table(RESULTS_DIR, SEED)

## Per-epoch old-task accuracy (rows = epochs, columns = arms): when does the damage happen?

In [ ]:
H.per_epoch_table(RESULTS_DIR, SEED)